# Data landscape

What we hold, country by country, and what matters about it.

## Section A — Country Attributes

The legal attributes an NPL buyer cares about most in each country: the ones that change
**whether** a debt can still be collected (limitation), **how** (court, bailiff, an
enforcement authority) and **whether it is frozen** (insolvency or a debt-relief scheme).
Not exhaustive: the main few per country.

**Named the way each country names it.** `attribute` is the term that country's market
uses, in English, and `local_term` is the word itself. Sweden says bankruptcy (konkurs) and
Germany insolvency (Insolvenz); Italy says injunction (decreto ingiuntivo), not "titled".
So a country that never says "insolvency" has no `insolvency_flag`. A consumer-only
debt-relief scheme (skuldsanering, WSNP, surendettement…) is its own row, because it works
differently from bankruptcy.

One row per (country, attribute); `meaning` says what it is in plain words.

**`static_or_variable`** says whether the value can change while we own the loan.
`static` is fixed once known, or measured at a fixed date: the default date, a birth date,
and so age at cut-off. `variable` can change after registration: a flag that switches on,
a limitation date that restarts, a balance that is paid down, a phone number that is added.

**How solid it is.** `legal_source` says where each row comes from:

- `web 2026-11-02`: checked against legal sources today (sources are listed at the end
  of this section).
- `general`: standard practice, not checked line by line.

Neither is legal advice. Check with counsel before a number drives a price.

**`used_in_npl_underwriting`** says whether the underwriting models actually use the
attribute. Measured 2026-11-02 across its 18 live deals: `yes` (built and matched,
segmented or modelled on, naming the deals), `built only` (the column is made but nothing
reads it), `no`, or `no deal` (npl-underwriting has no deal in that country). Nine of the
16 countries have no deal, and **no deal uses a limitation date**.

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 250)

In [ ]:
# (country, attribute, local_term, meaning, why_it_matters, legal_source)
# `attribute` is named the way that country's market names it, in English; `local_term`
# is the word itself. So Sweden says bankruptcy (konkurs), Germany insolvency (Insolvenz):
# a country that does not call it "insolvency" does not get an insolvency_flag.
ROWS = [
    # --- Nordics ---------------------------------------------------------------------
    ("Sweden", "limitation_date", "preskription", "Debt time-barred: 3 years for a consumer claim, 10 otherwise; a reminder or acknowledgement restarts it", "Past it, the claim cannot be enforced", "general"),
    ("Sweden", "enforcement_authority_flag", "Kronofogden, betalningsföreläggande", "Claim filed with the Enforcement Authority as an order to pay", "The main collection route once amicable fails", "general"),
    ("Sweden", "debt_restructuring_flag", "skuldsanering", "Person in debt restructuring", "Payments capped for years, the rest written off", "general"),
    ("Sweden", "bankruptcy_flag", "konkurs", "Debtor bankrupt, person or company", "Collection stops; recovery is a dividend from the estate", "general"),
    ("Finland", "limitation_date", "vanhentuminen", "Ordinary limitation: 3 years, restarted by a reminder or acknowledgement", "Must be interrupted to keep the debt alive", "web 2026-11-02"),
    ("Finland", "final_limitation_date", "lopullinen vanhentuminen", "Final limitation: 15 years from a judgment, 20 without one; cannot be interrupted", "A hard end date for collecting at all", "web 2026-11-02"),
    ("Finland", "enforcement_flag", "ulosotto", "Case with the Enforcement Authority", "Wage and asset attachment run through it", "general"),
    ("Finland", "debt_adjustment_flag", "velkajärjestely", "Person in court debt adjustment", "Payments set by a plan, the rest written off", "general"),
    ("Finland", "bankruptcy_flag", "konkurssi", "Debtor bankrupt, person or company", "Collection stops; recovery is a dividend from the estate", "general"),
    ("Norway", "limitation_date", "foreldelse", "Limitation: 3 years, restarted by acknowledgement or legal steps", "Past it, the claim cannot be enforced", "general"),
    ("Norway", "attachment_flag", "utlegg", "Attachment order obtained via the enforcement officer (namsmann)", "Secures wage or asset deductions", "general"),
    ("Norway", "debt_settlement_flag", "gjeldsordning", "Person in debt settlement", "Collection frozen; plan payments only", "general"),
    ("Norway", "bankruptcy_flag", "konkurs", "Debtor bankrupt, person or company", "Collection stops; recovery is a dividend from the estate", "general"),
    ("Denmark", "limitation_date", "forældelse", "Limitation: 3 years; 10 once a judgment or acknowledgement exists", "Past it, the claim cannot be enforced", "general"),
    ("Denmark", "bailiff_court_flag", "fogedret", "Case at the enforcement court", "Where asset or wage attachment is decided", "general"),
    ("Denmark", "debt_relief_flag", "gældssanering", "Person granted debt relief", "Most of the claim written off", "general"),
    ("Denmark", "bankruptcy_flag", "konkurs", "Debtor bankrupt, person or company", "Collection stops; recovery is a dividend from the estate", "general"),
    # --- DACH ------------------------------------------------------------------------
    ("Germany", "titled_flag", "tituliert, Vollstreckungsbescheid", "Claim holds an enforceable title", "Titled claims last 30 years and can be enforced", "general"),
    ("Germany", "limitation_date", "Verjährung", "Limitation: 3 years, running from the end of the year the claim arose", "Untitled claims expire fast", "general"),
    ("Germany", "insolvency_flag", "Insolvenz (Regelinsolvenz / Privatinsolvenz)", "Debtor insolvent: a company, or a person aiming at discharge (Restschuldbefreiung)", "Collection stops; a person's claim is usually discharged", "general"),
    ("Germany", "insolvency_date", "Insolvenzeröffnung", "Date the insolvency proceedings opened", "Starts the clock to discharge, usually 3 years for a person", "general"),
    ("Germany", "insolvency_age", "(derived)", "Time from the insolvency date to the cut-off", "Old insolvencies are near discharge, so near zero value", "general"),
    ("Germany", "title_date", "Datum des Titels / Vollstreckungsbescheid", "Date the enforceable title was obtained", "The 30-year title clock starts here", "general"),
    ("Germany", "title_age", "(derived)", "Time from the title date to the cut-off", "Fresh titles are still being enforced; old ones were worked already", "general"),
    ("Germany", "address_unknown_flag", "unbekannt verzogen", "Debtor moved and the new address is unknown", "No letter or bailiff can reach them until traced", "general"),
    ("Austria", "titled_flag", "Exekutionstitel", "Claim holds an enforceable title", "Long limitation once titled; enforcement possible", "general"),
    ("Austria", "limitation_date", "Verjährung", "Limitation: 3 years for most consumer claims, e.g. goods and services (§ 1486 ABGB); 30 years in general and once titled", "Past it, the claim cannot be enforced", "general"),
    ("Austria", "insolvency_flag", "Insolvenz (Konkurs / Schuldenregulierungsverfahren)", "Debtor insolvent: a company, or a person in personal bankruptcy ('Privatkonkurs')", "Collection stops; plan payments only", "general"),
    ("Switzerland", "debt_enforcement_stage", "Betreibung: Zahlungsbefehl → Pfändung", "Stage of statutory debt enforcement: payment order, then attachment", "Switzerland collects through this public process", "general"),
    ("Switzerland", "loss_certificate_flag", "Verlustschein", "Attachment found too little, so a loss certificate was issued", "The debt then lasts 20 years from the certificate (Art. 149a SchKG)", "web 2026-11-02"),
    ("Switzerland", "limitation_date", "Verjährung", "Limitation: 10 years in general, 5 for periodic payments", "Past it, the claim cannot be enforced", "general"),
    ("Switzerland", "bankruptcy_flag", "Konkurs / Privatkonkurs", "Debtor bankrupt: a company, or a person declaring personal bankruptcy", "No discharge for a person: the unpaid part becomes a loss certificate", "general"),
    # --- Benelux & France ------------------------------------------------------------
    ("Netherlands", "limitation_date", "verjaring", "Limitation: 5 years in general, 2 for consumer purchases; 20 once a judgment exists", "Past it, the claim cannot be enforced", "general"),
    ("Netherlands", "statutory_debt_restructuring_flag", "WSNP", "Person in statutory debt restructuring", "Collection frozen, usually 18 months to 3 years, then discharged", "general"),
    ("Netherlands", "bailiff_flag", "deurwaarder, beslag", "Bailiff involved, or attachment made", "Only a bailiff can enforce a judgment", "general"),
    ("Netherlands", "bankruptcy_flag", "faillissement", "Debtor bankrupt, person or company", "Collection stops; recovery is a dividend from the estate", "general"),
    ("Belgium", "collective_debt_settlement_flag", "collectieve schuldenregeling / règlement collectif de dettes", "Person in collective debt settlement", "Collection frozen; a court-set plan", "general"),
    ("Belgium", "judgment_flag", "vonnis / jugement", "Judgment obtained", "Needed before a bailiff can enforce", "general"),
    ("Belgium", "limitation_date", "verjaring / prescription", "Limitation: 10 years for personal claims, 5 for periodic sums", "Past it, the claim cannot be enforced", "general"),
    ("Belgium", "bankruptcy_flag", "faillissement / faillite", "Business debtor bankrupt: companies and the self-employed", "Collection stops; recovery is a dividend from the estate", "general"),
    ("France", "limitation_date", "prescription biennale", "Consumer claim lapses after 2 years (Code de la consommation L218-2)", "Very short: untitled consumer debt dies fast", "web 2026-11-02"),
    ("France", "enforceable_title_flag", "titre exécutoire, e.g. injonction de payer", "Enforceable title held", "Lasts 10 years; enforcement then possible", "general"),
    ("France", "overindebtedness_flag", "surendettement", "Person's over-indebtedness file at the Banque de France", "Collection frozen; a plan or write-off follows", "general"),
    ("France", "collective_proceedings_flag", "procédure collective: sauvegarde / redressement / liquidation judiciaire", "Business debtor in court insolvency proceedings", "Claim must be filed in the proceeding; recovery is usually low", "general"),
    # --- Southern Europe -------------------------------------------------------------
    ("Spain", "limitation_date", "prescripción", "Limitation: 5 years for personal claims (Art. 1964 Civil Code, since Oct 2015; 15 before)", "Older claims hit the transition date of 7 Oct 2020", "web 2026-11-02"),
    ("Spain", "judicialised_flag", "judicializado, monitorio", "Court claim filed, typically a payment procedure", "Filing interrupts limitation; the route to a title", "general"),
    ("Spain", "insolvency_flag", "concurso de acreedores, Ley de Segunda Oportunidad", "Debtor insolvent, person or company; a person can be discharged", "Claim may be discharged", "general"),
    ("Portugal", "judicialised_flag", "judicializado: processo executivo / injunção", "Debt in court: enforcement proceedings or payment injunction", "Most recovery comes through court; drives timing", "general"),
    ("Portugal", "limitation_date", "prescrição", "Limitation: 5 years per instalment for instalment loans; 20 years for cards and overdrafts (Civil Code arts. 309-310)", "Which one applies depends on the product", "web 2026-11-02"),
    ("Portugal", "insolvency_flag", "insolvência, exoneração do passivo restante", "Debtor insolvent, person or company; a person is often discharged of the rest", "Claim usually written off after the plan", "general"),
    ("Italy", "injunction_flag", "decreto ingiuntivo", "Payment injunction obtained", "The usual route to an enforceable title", "general"),
    ("Italy", "injunction_date", "data del decreto ingiuntivo", "Date the payment injunction was granted", "Starts the 10-year clock on the judgment", "general"),
    ("Italy", "injunction_age", "(derived)", "Time from the injunction date to the cut-off", "How long the debt has been enforceable without paying off", "general"),
    ("Italy", "enforcement_flag", "esecuzione, pignoramento", "Enforcement under way, e.g. attachment", "Long and variable timelines", "general"),
    ("Italy", "limitation_date", "prescrizione", "Limitation: 10 years in general, 5 for periodic payments", "Past it, the claim cannot be enforced", "general"),
    ("Italy", "judicial_liquidation_flag", "liquidazione giudiziale (formerly fallimento)", "Company in court liquidation", "Collection stops; recovery is a dividend from the estate", "general"),
    ("Italy", "over_indebtedness_flag", "sovraindebitamento", "Person or small business in an over-indebtedness procedure", "Collection frozen; a plan or write-off follows", "general"),
    ("Greece", "debt_protection_flag", "Katseli law, out-of-court mechanism", "Debtor under a debt-settlement law or plan", "Payments set by law or plan", "general"),
    ("Greece", "payment_order_flag", "diatagi pliromis", "Payment order obtained", "Route to enforcement", "general"),
    ("Greece", "bankruptcy_flag", "ptochefsi (Law 4738/2020)", "Debtor bankrupt, person or company", "Collection stops; discharge possible after the procedure", "general"),
    # --- CEE & UK --------------------------------------------------------------------
    ("Poland", "limitation_date", "przedawnienie", "Limitation: 6 years (3 for business or periodic), ending on the last day of the calendar year; since July 2018", "Courts check it themselves on consumer claims", "web 2026-11-02"),
    ("Poland", "court_order_flag", "nakaz zapłaty, EPU", "Court payment order obtained, including via the e-court", "The usual route to a title", "general"),
    ("Poland", "bailiff_flag", "komornik", "With a court bailiff", "Enforcement runs through the bailiff", "general"),
    ("Poland", "bankruptcy_flag", "upadłość / upadłość konsumencka", "Debtor bankrupt: a company, or a person (consumer bankruptcy)", "Collection stops; a person's claim is usually written off after the plan", "general"),
    ("United Kingdom", "statute_barred_flag", "statute-barred", "Past the 6-year limitation (5 in Scotland) with no payment or acknowledgement", "Cannot be enforced in court", "general"),
    ("United Kingdom", "county_court_judgment_flag", "CCJ", "County Court Judgment obtained", "Enforceable judgment, valid 6 years", "general"),
    ("United Kingdom", "individual_voluntary_arrangement_flag", "IVA / DRO", "Person in an Individual Voluntary Arrangement or a Debt Relief Order", "Payments set by the arrangement; usually 5 years", "general"),
    ("United Kingdom", "debt_management_company_flag", "DMC", "Borrower uses a debt management company that negotiates and pays on their behalf", "Payments come via the DMC, usually small but steady", "general"),
    ("United Kingdom", "unenforceable_flag", "unenforceable (CCA)", "Agreement cannot be enforced in court, e.g. a Consumer Credit Act defect", "Only voluntary payments; no court route", "general"),
    ("United Kingdom", "bankruptcy_flag", "bankruptcy (person); administration / liquidation (company)", "Debtor bankrupt, or a company in administration or liquidation", "Collection stops; recovery is a dividend, often small", "general"),
]

# Whether npl-underwriting actually uses each attribute: a deal builds the column AND
# matches, segments, filters or models on it. Measured 2026-11-02 over the 18 live deals
# in npl-underwriting/deals and its shared packages (which use no legal column at all).
#   yes        built and used, with the stage it is used in
#   built only the column is made but nothing downstream reads it
#   no         a deal in this country exists, but none carries it
#   no deal    npl-underwriting has no deal in this country
# Limitation is used nowhere: no deal builds a limitation or time-barred date.
NO_DEAL = "no deal"
USED = {
    ("Sweden", "debt_restructuring_flag"): "yes: matching and benchmark filter (ember dr_flag)",
    ("Sweden", "enforcement_authority_flag"): "built only (ember bailiff_flag; its matching combinations are commented out)",
    ("Sweden", "bankruptcy_flag"): "built only (ember-fresh bankruptcy_flag)",
    ("Denmark", "bailiff_court_flag"): "built only (bnp_dk bailiff_flag)",
    ("Denmark", "debt_relief_flag"): "built only (bnp_dk dr_flag)",
    ("Germany", "titled_flag"): "yes: matching (ge25020, ge26004, ge26007, ge26009, targobank)",
    ("Germany", "insolvency_flag"): "yes: matching (ge26004, ge26007, ge26009, ge26011, korund, targobank)",
    ("Germany", "insolvency_date"): "yes: as insolvency_age (date present in 8 deals)",
    ("Germany", "insolvency_age"): "yes: matching (ge26004, ge26007-santander, ge26011-db-ff)",
    ("Germany", "title_date"): "yes: as title_age (date present in 6 deals)",
    ("Germany", "title_age"): "yes: matching (ge26007-santander)",
    ("Germany", "address_unknown_flag"): "yes: matching (ge26004 unknown_moved_flag)",
    ("France", "enforceable_title_flag"): "yes: reporting only, not matching (iqera enforcement_title_status)",
    ("France", "overindebtedness_flag"): "partly: a 'BDF' mandate type in reporting, no flag of its own (iqera)",
    ("Italy", "injunction_flag"): "yes: modelling stages (banka-ifis decreto ingiuntivo)",
    ("Italy", "injunction_date"): "yes: as injunction_age (banka-ifis)",
    ("Italy", "injunction_age"): "yes: modelling (banka-ifis title_age_months)",
    ("Italy", "enforcement_flag"): "yes: modelling stages (banka-ifis precetto, pignoramento)",
    ("Italy", "judicial_liquidation_flag"): "yes: modelling, as generic insolvency stages (banka-ifis concorsuale)",
    ("Italy", "over_indebtedness_flag"): "yes: modelling segment (banka-ifis sovraindebitamento)",
    ("Poland", "bailiff_flag"): "yes: benchmark filter (lok-pl26003 collection_phase)",
    ("United Kingdom", "county_court_judgment_flag"): "yes: benchmark filter, as judgement_date (tesco)",
    ("United Kingdom", "individual_voluntary_arrangement_flag"): "yes: segment and matching (tesco)",
    ("United Kingdom", "debt_management_company_flag"): "yes: matching and reporting (tesco dmc_flag)",
    ("United Kingdom", "unenforceable_flag"): "yes: matching and reporting (tesco ue_flag, as litigation_restricted)",
}
# Static: fixed once known, or measured at a fixed date (age to cut-off is static because
# the cut-off is). Everything else is variable: it can change while we own the loan.
# A limitation date is variable because a payment or acknowledgement restarts it; a final
# limitation date is static because nothing can.
STATIC = {
    # Section A
    "final_limitation_date", "insolvency_date", "insolvency_age", "title_date", "title_age",
    "injunction_date", "injunction_age", "unenforceable_flag",
    # Section B
    "age", "gender", "individual_vs_corporate", "has_codebtor", "has_guarantor",
    "has_multiple_cases", "origination_date", "default_date", "debt_age", "cut_off_date",
    "registration_date", "prior_capex_date", "purchase_price", "prior_purchase_price", "country", "case_count_at_cut_off", "case_count_at_registration",
    "collection_months_pre_cut_off", "collection_months_pre_registration",
    "paying_case_count_pre_cut_off", "paying_case_count_pre_registration", "registration_balance", "cut_off_balance",
    "registration_principal", "cut_off_principal", "registration_interest", "cut_off_interest",
    "registration_fee", "cut_off_fee", "default_balance", "original_amount", "interest_rate",
    "product_type", "seller_name", "pricing_segment", "deal_flow_type", "currency", "days_past_due",
    "npl_backstop_flag", "has_collateral", "collateral_type",
}


def add_static_or_variable(df):
    """The static_or_variable column, placed right after `attribute`."""
    df.insert(2, "static_or_variable", df.attribute.map(lambda a: "static" if a in STATIC else "variable"))
    return df


HAS_DEAL = {"Sweden", "Denmark", "Germany", "France", "Italy", "Poland", "United Kingdom"}

country_attributes = pd.DataFrame(
    ROWS, columns=["country", "attribute", "local_term", "meaning", "why_it_matters", "legal_source"]
)
country_attributes["used_in_npl_underwriting"] = [
    USED.get((c, a), "no" if c in HAS_DEAL else NO_DEAL)
    for c, a in zip(country_attributes.country, country_attributes.attribute)
]
add_static_or_variable(country_attributes)
used = country_attributes.used_in_npl_underwriting.str.startswith(("yes", "partly"))
print(f"{country_attributes.country.nunique()} countries, {len(country_attributes)} attributes; "
      f"{used.sum()} used by npl-underwriting, "
      f"{country_attributes.used_in_npl_underwriting.str.startswith('built').sum()} built only; "
      f"{(country_attributes.static_or_variable == 'static').sum()} static")
country_attributes

Every country at a glance: its attributes on one row.


In [ ]:
(
    country_attributes.groupby("country", sort=False)["attribute"]
    .agg(lambda a: ", ".join(a))
    .rename("attributes")
    .to_frame()
)


One country at a time:

Sources for the rows marked `web 2026-11-02`:

- Finland: [Act on the Limitation of Debts 728/2003 (Finlex)](https://www.finlex.fi/en/legislation/translations/2003/eng/728) · [Legal Aid and Public Guardianship: time-barred debts](https://oikeuspalveluvirasto.fi/en/financial-and-debt-counselling/debt-collection-and-payment-defaults/time-barred-debts/)
- France: [Prescription biennale, L218-2](https://consultation.avocat.fr/blog/florent-delpoux/article-35412-la-prescription-biennale-en-matiere-de-credit.html)
- Spain: [Andersen: 7 Oct 2020 deadline](https://es.andersen.com/en/blog/2020/02/28/octubre-de-2020-fecha-limite-para-reclamar-deudas-de-2015/) · [Clifford Chance briefing](https://www.cliffordchance.com/content/dam/cliffordchance/briefings/2020/01/Statute-limitations-personal.pdf)
- Portugal: [DECO Proteste: prescrição de dívidas](https://www.deco.proteste.pt/familia-consumo/orcamento-familiar/dicas/prescricao-dividas-partir-quando-fico-livre-pagar)
- Poland: [Dudkowiak: limitation periods in Poland](https://www.dudkowiak.com/debt-collection-in-poland/limitation-of-claims-in-poland.html) · [In Principle: 2018 changes](https://codozasady.pl/en/p/major-changes-to-rules-on-statutory-time-limits-on-claims)
- Switzerland: [Beobachter: Verjähren Verlustscheine?](https://www.beobachter.ch/schulden-betreibungen/verjahren-verlustscheine)

## Section B — General Attributes

The attributes that matter on **every** tape, whatever the country: who the borrower is,
the key dates, the balance split, the product, any collateral and the collection history.
Same columns as Section A, with `country` = `All` and `legal_source` = `Not relevant`.
`static_or_variable` follows the same rule.

`local_term` is what the column is typically called on a tape, in
**Swedish · German · Portuguese · Italian · Spanish**. `(derived)` means nobody sends it:
it is built from other columns.

**`used_in_npl_underwriting`** starts with **Yes** or **No**, then says how. Measured
2026-11-02 across its 18 live deals and shared packages:

- **Yes**: in the shared schema (`npl-schema`), or matched, segmented, modelled or filtered on.
- **No (built only)**: a deal makes the column but nothing reads it.

Two caveats from that search. Only 5 of the 18 deals run on today's shared packages; the
other 13 are frozen on the old version. And a column listed in a reporting strat counts
only if the deal actually builds it, because the strat code skips missing columns silently.

In [ ]:
# (attribute, local_term SV · DE · PT · IT · ES, meaning, why_it_matters, used_in_npl_underwriting)
GENERAL = [
    # --- Borrower ----------------------------------------------------------------------
    ("age", "ålder / födelseår · Alter / Geburtsdatum · idade / data de nascimento · età / data di nascita · edad / fecha de nacimiento", "Borrower's age, usually from a birth date or year", "Older and younger debtors pay differently; a common matching feature", "Yes: schema (customer_age, optional); matched in ge26011"),
    ("gender", "kön · Geschlecht · sexo · sesso · sexo", "Borrower's gender", "Weak predictor; often stripped as personal data", "Yes: matched in one deal (ge26007 is_male)"),
    ("town", "ort · Ort · localidade · comune · localidad", "Borrower's town", "Local economy and court location; personal data, so coarsen", "No"),
    ("city_region", "postnummer / län · PLZ / Bundesland · código postal / distrito · CAP / provincia · código postal / provincia", "Region or postcode area", "Regional recovery differences; postcode is personal data, keep only the district", "No (built only: ember-fresh region). banka-ifis uses the court's province, not the debtor's"),
    ("has_phone_number", "telefon · Telefon · telefone · telefono · teléfono", "A phone number is on file (yes/no, not the number)", "Contactable debtors pay more; the flag is safe to keep", "No"),
    ("has_email", "e-post · E-Mail · e-mail · e-mail · correo electrónico", "An email address is on file (yes/no)", "Same: contactability, without the personal data", "No"),
    ("individual_vs_corporate", "privatperson / företag · Privatperson / Firma · particular / empresa · persona fisica / giuridica · particular / empresa", "Loan holder is a person or a company", "Companies and people recover very differently", "Yes: schema (customer_type, optional); benchmark filter (targobank); modelled (banka-ifis)"),
    ("deceased_flag", "avliden · verstorben · falecido · deceduto · fallecido", "Borrower has died", "Recovery then runs against the estate, usually little", "Yes: modelled (banka-ifis; zero capacity). ember builds it only"),
    ("has_codebtor", "medlåntagare · Mitschuldner · co-devedor · cointestatario · codeudor", "A second borrower is jointly liable for the loan", "Two people to collect from", "No"),
    ("has_guarantor", "borgensman · Bürge · fiador · fideiussore / garante · avalista / fiador", "A guarantor has promised to pay if the borrower does not", "A second source of payment, through a separate claim", "Yes: modelled (banka-ifis guarantor streams, secured segment)"),
    ("has_multiple_cases", "(derived)", "Borrower has more than one case in the book", "Debtors with several debts behave differently", "Yes: matching (lok-pl26003 multi_debtor_flag; the code does not say what 'multi' means)"),
    ("employment_type", "anställningsform · Beschäftigungsart · situação profissional · tipo di occupazione · situación laboral", "Employed, self-employed, pensioner, unemployed…", "Decides whether wages or a pension can be attached", "Yes: modelled (banka-ifis employability segments). ember builds it only"),
    ("net_income", "inkomst · Einkommen · rendimento · reddito · ingresos", "Borrower's net income", "Caps what wage attachment can recover", "Yes: modelled (banka-ifis wage and pension attachment)"),
    ("credit_score", "kreditbetyg · Bonität / SCHUFA · score de crédito · merito creditizio · scoring", "Seller's or bureau credit score", "Ability-to-pay signal", "No (built only: ember, ember-fresh)"),
    ("vulnerable_flag", "sårbar kund · schutzbedürftig · vulnerável · vulnerabile · vulnerable", "Borrower marked vulnerable", "Restricts how collection may be done", "No (built only: tesco)"),
    # --- Dates -------------------------------------------------------------------------
    ("origination_date", "utbetalningsdatum · Abschlussdatum · data de originação · data di erogazione · fecha de formalización", "Date the loan was first granted", "Loan vintage; some rules depend on it", "Yes: schema (optional); reporting only (tesco, targobank)"),
    ("default_date", "förfallodatum / uppsägningsdatum · Kündigungsdatum · data de incumprimento · data di default / decadenza · fecha de impago", "Date the loan went non-performing", "Debt age starts here; older debt pays less", "Yes: schema (optional); drives debt_age everywhere"),
    ("debt_age", "(derived)", "Time from default date to registration", "One of the strongest predictors of recovery", "Yes: schema (optional); matched in almost every deal"),
    ("cut_off_date", "brytdatum · Stichtag · data de corte · data di cut-off · fecha de corte", "Date the economics pass to the buyer: collections after it are ours", "Splits the seller's cash from ours", "Yes: insolvency and title age measured to it (ge26004, ge26007)"),
    ("registration_date", "registreringsdatum · Übertragungsdatum · data de registo · data di cessione · fecha de cesión", "Date the buyer legally owns the loan and starts collecting", "Time zero for every collection curve", "Yes: schema (mandatory)"),
    ("prior_capex_date", "förvärvsdatum · Erwerbsdatum · data de aquisição · data di acquisto · fecha de adquisición", "Secondary deals only (a debt buyer reselling a book it bought): the date that seller bought it", "Shows how long another collector has already worked it", "Yes: cohorts (poplar target model). banka-ifis builds it only"),
    ("purchase_price", "köpeskilling · Kaufpreis · preço de compra · prezzo di acquisto · precio de compra", "What we pay for the book, often quoted as a % of its balance; set per deal, not per loan", "The number underwriting exists to set", "No: it is the output, not an input. Curves are turned into a price outside the code"),
    ("prior_purchase_price", "tidigare köpeskilling · früherer Kaufpreis · preço de compra anterior · prezzo di acquisto precedente · precio de compra anterior", "Secondary deals only (a debt buyer reselling a book it bought): what that seller paid for it", "Shows what the last buyer thought it was worth", "No: no deal has it"),
    ("last_payment_date", "senaste betalning · letzte Zahlung · data do último pagamento · data ultimo pagamento · fecha último pago", "When the borrower last paid before registration", "Recent payers keep paying", "Yes: payment recency (payment_category) matched in most deals; tesco also days_since_last_payment"),
    # --- Balances ----------------------------------------------------------------------
    ("registration_balance", "skuld vid registrering · Forderung bei Übertragung · saldo na cessão · saldo alla cessione · saldo a la cesión", "Total owed when the buyer took the loan", "The base every recovery rate is divided by", "Yes: schema (mandatory, > 0); matched in every deal"),
    ("default_balance", "skuld vid uppsägning · Saldo bei Kündigung · saldo em incumprimento · saldo al default · saldo al impago", "Total owed on the default date", "Shows how much interest and fees were added since", "Yes: filter > 0 (bnp_dk); ember-fresh cleaning"),
    ("legal_claim_amount", "fordringsbelopp · Gesamtforderung · montante reclamado · pretesa creditoria · importe reclamado", "Amount claimed in court, which can differ from the balance", "What a court can actually award", "Yes: reporting (iqera total_claim); segments (banka-ifis)"),
    ("cut_off_balance", "skuld vid brytdatum · Forderung zum Stichtag · saldo na data de corte · saldo al cut-off · saldo a fecha de corte", "Total owed on the cut-off date", "What the price is usually quoted on", "No"),
    ("registration_principal", "kapital · Hauptforderung · capital · capitale · principal", "Principal part of the registration balance", "Principal recovers differently from interest and fees", "No (built only: norion, banka-ifis, as parts of the balance)"),
    ("cut_off_principal", "kapital vid brytdatum · Hauptforderung zum Stichtag · capital na data de corte · capitale al cut-off · principal a fecha de corte", "Principal on the cut-off date", "Same split, on the pricing date", "No"),
    ("registration_interest", "ränta · Zinsen · juros · interessi · intereses", "Interest part of the registration balance", "Often the first part written off", "No (built only: norion, banka-ifis)"),
    ("cut_off_interest", "ränta vid brytdatum · Zinsen zum Stichtag · juros na data de corte · interessi al cut-off · intereses a fecha de corte", "Interest on the cut-off date", "Same split, on the pricing date", "No"),
    ("registration_fee", "avgifter · Gebühren / Kosten · comissões / despesas · spese / commissioni · comisiones / gastos", "Fees and costs part of the registration balance", "Often not legally collectable in full", "No (built only: norion)"),
    ("cut_off_fee", "avgifter vid brytdatum · Gebühren zum Stichtag · despesas na data de corte · spese al cut-off · gastos a fecha de corte", "Fees and costs on the cut-off date", "Same split, on the pricing date", "No"),
    ("original_amount", "ursprungligt belopp · Ursprungsbetrag · montante inicial · importo originario · importe inicial", "Amount originally lent", "How much of the loan was ever repaid", "No (built only: ember, banka-ifis)"),
    ("interest_rate", "räntesats · Zinssatz · taxa de juro · tasso di interesse · tipo de interés", "Contract interest rate", "How fast the balance grows", "No (built only: banka-ifis)"),
    # --- Product and deal --------------------------------------------------------------
    ("product_type", "produkt · Produkt · produto · prodotto · producto", "Loan type: personal loan, card, overdraft, telecom…", "Products recover very differently", "Yes: schema (optional); matched in 4 German deals; target-model feature"),
    ("seller_name", "säljare / borgenär · Verkäufer / Gläubiger · cedente · cedente · cedente", "Who sold the book", "Seller practice shapes the book; used to pick benchmarks", "Yes: benchmark filter (tesco, ge26004, iqera, ember); curves split by it (ember)"),
    ("pricing_segment", "(derived)", "The segment a deal is priced in, e.g. sweden_npl, IVA", "Which benchmarks are allowed to match", "Yes: schema (optional); filter in every matching deal"),
    ("deal_flow_type", "affärstyp · Transaktionsart · tipo de operação · tipo di operazione · tipo de operación", "One-off (a book sold once) or forward flow (new defaults sold every month or quarter)", "Forward-flow books are younger and pay more", "Yes: benchmark filter (uk26006)"),
    ("npl_backstop_flag", "(derived)", "Loan falls under the EU NPL backstop: granted after 26 Apr 2019 and in default long enough that the bank must fully provision it", "Banks sell these to escape the capital charge", "Yes: reporting (tesco, targobank)"),
    ("country", "land · Land · país · paese · país", "Country the loan sits in, which sets the legal regime", "Picks the Section A rules and the benchmarks that apply", "Yes: schema (optional, target model); filter in norion-sweden"),
    ("case_count_at_cut_off", "antal ärenden vid brytdatum · Fallzahl zum Stichtag · n.º de processos na data de corte · numero pratiche al cut-off · n.º de expedientes a fecha de corte", "Number of cases in the book on the cut-off date; set per deal", "What the price is quoted on; with the registration count, shows how many cases dropped out in between", "No"),
    ("case_count_at_registration", "antal ärenden vid registrering · Fallzahl bei Übertragung · n.º de processos na cessão · numero pratiche alla cessione · n.º de expedientes a la cesión", "Number of cases we actually took over at registration; set per deal", "The base for per-case figures, e.g. average balance", "Yes: schema (n_accounts per registration cohort, target model and matching output)"),
    ("currency", "valuta · Währung · moeda · valuta · moneda", "Currency of the balances", "Never add balances across currencies", "Yes: schema (optional, portfolio level)"),
    ("account_status", "öppen / stängd · offen / geschlossen · aberto / fechado · aperto / chiuso · abierto / cerrado", "Account still open, or closed (paid, settled, written off)", "Closed accounts carry no future cash", "Yes: schema (is_open, mandatory for the target model); norion, iqera"),
    ("days_past_due", "dagar efter förfall · Verzugstage · dias de atraso · giorni di scaduto · días de impago", "Days past due", "How deep in arrears", "No (built only: ember-fresh dpd_bucket)"),
    # --- Collateral --------------------------------------------------------------------
    ("has_collateral", "säkerhet · Sicherheit · garantia · garanzia reale · garantía", "Loan is secured", "Secured loans recover through the asset", "Yes: banka-ifis secured segment"),
    ("collateral_value", "säkerhetens värde · Sicherheitenwert · valor da garantia · valore della garanzia · valor de la garantía", "Value of the collateral", "Caps what the asset can return", "Yes: banka-ifis property valuation"),
    ("collateral_type", "typ av säkerhet · Art der Sicherheit · tipo de garantia · tipo di garanzia · tipo de garantía", "What the collateral is: property, car, land…", "Different assets sell at different speeds", "Yes: banka-ifis (residential, land…)"),
    # --- Collections -------------------------------------------------------------------
    ("monthly_collection_sum", "(derived) inbetalningar · Zahlungseingänge · pagamentos · incassi · cobros", "Total collected per month, from the payment rows", "The collection curve itself", "Yes: sum and mean of collections per window, matched in every deal"),
    ("monthly_collection_count", "(derived) antal betalningar · Anzahl Zahlungen · n.º de pagamentos · numero incassi · n.º de pagos", "Number of payments per month", "How regularly the borrower pays", "Yes: payment count per window, matched in every deal"),
    ("collection_months_pre_cut_off", "(derived)", "Months of payment history the seller gives before the cut-off date", "How far back a collection curve can be read before we price", "Yes: sets which collection windows (collections_between_X_Y) a deal can match on"),
    ("collection_months_pre_registration", "(derived)", "Months of payment history before registration, including the gap between cut-off and registration", "The seller's cash, which never reaches us; the base for interim collections", "Yes: the pre-registration windows matched in most deals (payment_category, collections_between_X_Y)"),
    ("collection_months_post_registration", "(derived)", "Months of payments we have seen since registration", "How much of the forecast curve is already actual", "Yes: interim actuals (targobank liquidations_between_1_N, tesco Liq_1m)"),
    ("total_collections_pre_cut_off", "(derived)", "Total collected before the cut-off date, summed from the seller's own payment rows", "What the book had already paid the seller: the level a recovery curve starts from", "Yes: the pre-registration collection windows matched in most deals"),
    ("total_collections_post_registration_cat", "(derived)", "Total collected since registration, summed from the servicer's case-level payments (Intrum CAT)", "What the book has actually paid us, case by case", "Yes: interim actuals; FCT_CASE_COLLECTION feeds the servicing curves"),
    ("total_collections_post_registration_pam", "(derived)", "Total collected since registration, from the servicer's portfolio-level monthly report (Intrum PAM)", "The servicer's own total, to check the case-level one against", "No: portfolio level, outside npl-underwriting"),
    ("paying_case_count_pre_cut_off", "(derived)", "Number of cases that paid at least once before cut-off", "The payers in the book as priced", "Yes: made_payment_* features in matching"),
    ("paying_case_count_pre_registration", "(derived)", "Number of cases that paid at least once before registration", "Payers when we take over; recent payers keep paying", "Yes: made_payment_* and payment recency in matching"),
    ("paying_case_count_post_registration", "(derived)", "Number of cases that have paid since registration", "How many of the book are paying us", "No"),
    ("payer_flag", "(derived)", "Paid at least once before or after registration", "Payers are the book's value", "Yes, EDA only: shared stats, not matching"),
    ("payment_plan_flag", "avbetalningsplan · Ratenzahlung · plano de pagamentos · piano di rientro · plan de pagos", "Borrower is on a repayment plan", "Plans pay steadily but slowly", "Yes: banka-ifis plan segment"),
    ("third_party_managed_flag", "inkassobolag · Inkassobüro · agência de cobrança · agenzia di recupero · agencia de recobro", "Debt is with a collection agency now", "Shows collection effort under way", "Yes: matching (tesco dca_flag)"),
    ("current_servicer_name", "inkassoombud · Inkassodienstleister · servicer / gestor · servicer / gestore · servicer / gestor", "Who collects the debt today", "Servicers differ in how much they recover", "Yes: reporting (tesco dca_name); segment join (banka-ifis)"),
    ("prior_servicer_count", "(derived)", "How many collection agencies have already worked it", "Each failed placement means less left to collect", "Yes: matching and reporting (tesco n_dca_placements_new)"),
]

general_attributes = pd.DataFrame(
    [("All", a, t, m, w, "Not relevant", u) for a, t, m, w, u in GENERAL],
    columns=["country", "attribute", "local_term", "meaning", "why_it_matters", "legal_source", "used_in_npl_underwriting"],
)
add_static_or_variable(general_attributes)
used = general_attributes.used_in_npl_underwriting.str.startswith("Yes")
print(f"{len(general_attributes)} general attributes: {used.sum()} used by npl-underwriting, "
      f"{(~used).sum()} not (of which {general_attributes.used_in_npl_underwriting.str.contains('built only').sum()} built only); "
      f"{(general_attributes.static_or_variable == 'static').sum()} static, {(general_attributes.static_or_variable == 'variable').sum()} variable")
general_attributes

Sections A and B together, one table to filter or export:


In [ ]:
all_attributes = pd.concat([country_attributes, general_attributes], ignore_index=True)
all_attributes.groupby("country", sort=False).size().rename("attributes").to_frame().T


## Section C — Deals We Own

Every deal Brocc owns today, one row each, from a live PROD query:
[`deals_we_own.sql`](queries/deals_we_own.sql), which explains every rule below in its comments.

- **`deal_id`** is `VEHICLE-ID`. A Heimdall deal uses Intrum's id (`HEI-PT13982`); every
  other vehicle uses our Atlas id (`LOK-SE25008`, which Intrum calls SE25018).
- **`market`** is country × PAM product type (`SE · Bank Loan`), the two joined.
  `product_type` is PAM's own product, 14 of them. It is empty on the 12 LEG, GCB and ORG
  books PAM never reported. 13 deals were bought across two products and take the one
  holding the most principal.
- **The purchase described is the FIRST one**, and `acquired_by` says whose it was:
  - `Intrum` for Heimdall. Intrum bought these between 2001 and 2023, and Brocc bought
    Heimdall in December 2025.
  - `Brocc (Lokki)` for Lokki, our own vehicle, using PAM's record of our purchase.
  - `previous owner` for the 6 legacy Spanish and Italian books.
  - `Brocc` for GCB and ORG, which we bought first.

  One row never mixes two sources, so price ÷ balance is always one purchase's ratio.
- **`balance_at_acquisition`** is principal only for Heimdall, Lokki and the legacy books, because
  PAM and the servicers hold no interest, fees or costs at acquisition. It is the full
  balance (GBV: principal + interest + fees) only for GCB and ORG, which come from Atlas. `balance_basis` says which.
- **`registration_date`** is the first case registered on or after the capex month. The
  plain earliest date is misleading: a quarter of Heimdall cases were registered with
  Intrum before Intrum bought them, because it serviced them first.
- Money is in the deal's own currency (`_local`) and in EUR at the latest month-end rate
  (`_eur`). Only the `_eur` columns can be added across deals.

**Gaps, so nobody reads an empty cell as zero:**

- **No earlier purchase price** on the 6 legacy books (LEG).
- **No registration date** on LEG, GCB and ORG deals. These systems record none. It is
  also empty on about 700 Heimdall and 18 Lokki deals, either because all their cases
  predate capex or because no case is loaded yet.
- **No case count** on GCB and ORG, since Atlas does not hold one.
- **No price or balance** on ORG's two Equinox deals in Atlas.


In [ ]:
import sys

sys.path.insert(0, "../../packages")   # notebook sits in work/workshop/
import datalake

# Live PROD query, about a minute (most of it the registration date).
deals = datalake.q("work/workshop/queries/deals_we_own")
deals.columns = deals.columns.str.lower()

SHOW = [
    "deal_id", "country", "product_type", "market", "acquired_by", "deal_flow_type",
    "registration_date", "capex_date",
    "currency", "purchase_price_local", "purchase_price_eur", "cases_acquired",
    "balance_at_acquisition_local", "balance_at_acquisition_eur", "balance_basis",
]
print(f"{len(deals):,} deals we own, queried {pd.Timestamp.now():%Y-%m-%d}")
with pd.option_context("display.float_format", lambda v: f"{v:,.5f}"):
    display(deals[SHOW])

Per vehicle: how many deals, and how many have each figure.

In [ ]:
# min_count=1: a vehicle with no prices shows empty, not 0.
with pd.option_context("display.float_format", lambda v: f"{v:,.5f}"):
    display(
        deals.groupby(["vehicle", "acquired_by"])
        .agg(
            deals=("deal_id", "size"),
            with_registration_date=("registration_date", "count"),
            with_purchase_price=("purchase_price_local", "count"),
            with_case_count=("cases_acquired", "count"),
            purchase_price_eur=("purchase_price_eur", lambda v: v.sum(min_count=1)),
            balance_at_acquisition_eur=("balance_at_acquisition_eur", lambda v: v.sum(min_count=1)),
        )
    )

## Section D — What Data We Have

### D.1 Pre-registration data: the Asset Index

For every deal in Section C, whether the seller's files carry each attribute of
Sections A and B. The files are the sorted deliveries in [`asset-index/`](asset-index), one
folder per deal, named like Section C's `deal_id`.

`bridge_coverage` says whether the tape reaches Intrum's cases: **Found** when the deal's
`_meta/bridge.yaml` names a tape column that matches Intrum's case references, **Not Found**
when it names none or the deal has no `bridge.yaml` (5 deals: Intrum holds no cases for
them yet, or they are not Intrum-serviced).

Each attribute cell is one of:

- **Found**: a column in at least one of the deal's files carries it. For a derived
  attribute (debt age, collection months…) its inputs are there instead.
- **Not Found**: no readable file of the deal carries it.
- **Unreadable**: not found, but the deal has files nobody can open: 14 files are
  DRM-protected (encrypted with rights management). The attribute may be in those.
- **Not Relevant**: a Section A attribute of another country, e.g. a Dutch WSNP flag on a
  Swedish deal.
- **No Seller Tape**: we own the deal but hold no files for it.

**How "Found" is decided.** Only the header row of each sheet is read, never the values
below it. Each attribute has a pattern of the words a header uses for it, in English and
in the local terms of Sections A and B (the rules are in
[`asset_index_scan.py`](asset_index_scan.py)). So:

- **Solid:** a Found names a real column, listed in `d1_evidence` with the file it came from.
- **Shaky:**
  - A Not Found can be a column under a name no pattern knows.
  - A header can match the right word for the wrong thing. For example, a pricing sheet's
    "price" column is a bid, not the tape's price.
  - Every file in the folder counts, including Intrum's benchmark and case-mapping files,
    not only the seller's tape.

**Derived attributes** are Found when their inputs are:

- **Months and paying cases before cut-off:** a payment amount and a payment date or month.
- **Before registration:** the same, plus a registration date or a pre/post-registration flag.
- **After registration:** only where a file flags pre/post registration (Intrum's benchmark
  payments).
- **Case count at cut-off:** any case id, since a tape is one row per case.


In [ ]:
import yaml

import asset_index_scan

# Headers only, of every sheet of every spreadsheet and CSV: about 40 seconds.
headers, unreadable = asset_index_scan.headers()
d1_evidence = asset_index_scan.matches(headers)


def bridge_coverage(deal):
    """Share of Intrum's cases the tape's best column reaches; None where there is none.

    bridge.yaml writes coverage_pct out of 100 (0.70922 there means 0.7%, not 70%), so it
    is divided here, once, and everything downstream is a 0-1 share like every other rate
    in this workbook. Reading it raw is what made the 80% bridge gate admit 5 Portuguese
    deals sitting at 7-13%.
    """
    path = asset_index_scan.ROOT / deal / "_meta" / "bridge.yaml"
    if not path.exists():
        return None
    tape_bridge = (yaml.safe_load(path.read_text()) or {}).get("tape_bridge") or {}
    pct = tape_bridge.get("coverage_pct")
    return None if pct is None else pct / 100

COUNTRY_NAME = {
    "SE": "Sweden", "FI": "Finland", "NO": "Norway", "DK": "Denmark", "DE": "Germany",
    "AT": "Austria", "CH": "Switzerland", "NL": "Netherlands", "BE": "Belgium",
    "FR": "France", "ES": "Spain", "PT": "Portugal", "IT": "Italy", "GR": "Greece",
    "PL": "Poland", "GB": "United Kingdom",
}
ATTRIBUTES = list(dict.fromkeys(all_attributes.attribute))
# Which countries each Section A attribute belongs to; Section B's belong to all.
countries_of = country_attributes.groupby("attribute").country.agg(set).to_dict()

found = set(zip(d1_evidence.deal_id, d1_evidence.attribute))
locked = set(unreadable.deal_id)
in_folder = sorted(headers.deal_id.unique())
country = deals.set_index("deal_id").country.map(COUNTRY_NAME)


def cell(deal, attr):
    if attr in countries_of and country.get(deal) not in countries_of[attr]:
        return "Not Relevant"
    if (deal, attr) in found:
        return "Found"
    return "Unreadable" if deal in locked else "Not Found"


coverage_pct = {deal: bridge_coverage(deal) for deal in in_folder}
asset_index = pd.DataFrame(
    [[deal, "Not Found" if coverage_pct[deal] is None else "Found"]
     + [cell(deal, a) for a in ATTRIBUTES] for deal in in_folder],
    columns=["deal_id", "bridge_coverage"] + ATTRIBUTES,
)

# Every deal we own, with the asset-index picture where we have one.
# deal_country, because `country` is also one of the attributes.
d1 = (
    deals[["deal_id", "vehicle", "country"]].rename(columns={"country": "deal_country"})
    .merge(asset_index, on="deal_id", how="left")
)
d1[["bridge_coverage"] + ATTRIBUTES] = d1[["bridge_coverage"] + ATTRIBUTES].fillna("No Seller Tape")

print(f"{len(deals):,} deals we own; {len(in_folder)} have seller files in asset-index/, "
      f"{len(locked)} of them with an unreadable file ({len(unreadable)} files)")
d1[d1.deal_id.isin(in_folder)]

Per attribute, across the deals that have seller files: how many carry it.

In [ ]:
have_files = d1[d1.deal_id.isin(in_folder)]
coverage = (
    have_files[["bridge_coverage"] + ATTRIBUTES].apply(lambda col: col.value_counts()).T.fillna(0).astype(int)
    .reindex(columns=["Found", "Not Found", "Unreadable", "Not Relevant"], fill_value=0)
    .sort_values("Found", ascending=False)
)
coverage

The files that could not be read, and every match behind a Found (headers and the first file that has them):

In [ ]:
display(unreadable)
d1_evidence.sort_values(["deal_id", "attribute"])

### D.2 Data quality: how full is a Found column?

Same rows and columns as D.1. Where D.1 says **Found**, D.2 says how much of the column
holds real values:

- **`bridge_coverage`** is the share of Intrum's cases the tape reaches, as a decimal
  (0.098 = 9.8%), from `bridge.yaml`.
- **`Mainly Not Null (90%+)`**: at least 90% of the sheet's data rows hold a value.
- **`Mainly Null (10%+)`**: 10% or more are blank or a "not available". This includes a
  column that is entirely empty.

"Not available" covers `NaN`, `N/A`, `-`, `#N/A`, `unknown`, and the same words in Swedish,
Finnish, German, Dutch, French, Portuguese, Italian, Spanish, Polish and Greek (`saknas`,
`ei saatavilla`, `nicht verfügbar`, `não disponível`, `non disponibile`, `no disponible`…).
The full list is `NULL_LIKE` in [`asset_index_scan.py`](asset_index_scan.py).

How the rate is worked out:

- A data row is any row with at least one value, so blank rows left below a table do not
  count.
- Where several columns carry an attribute, the fullest one counts.
- A derived attribute is only as full as its emptiest input.
- The exact rates are in `d2_fill_rates`.

Cells D.1 did not find keep D.1's label (`Not Found`, `Unreadable`, `Not Relevant`,
`No Seller Tape`). Together, D.1 says whether the data **exists** and D.2 whether it is
**usable**.

Every row of every matched sheet is read in memory, and nothing but the rate comes out.
The rates are cached by file size and date in `.cache/`, so a rerun only reads files that
changed.


In [ ]:
# Reads every row of every matched sheet: ~10 minutes the first time, seconds from cache.
rated = asset_index_scan.fill_rates(asset_index_scan.hits(headers))
fill = asset_index_scan.attribute_fill(rated)


def quality_label(rate):
    if rate is None or pd.isna(rate):
        return "Unreadable"  # read for its headers, but its cells would not read
    return "Mainly Not Null (90%+)" if rate >= 0.9 else "Mainly Null (10%+)"


d2 = d1.copy().astype({"bridge_coverage": object})
for deal in in_folder:
    row = d2.deal_id == deal
    d2.loc[row, "bridge_coverage"] = (
        "Not Found" if coverage_pct[deal] is None else coverage_pct[deal]
    )
    for attr in ATTRIBUTES:
        if d1.loc[row, attr].iloc[0] == "Found":
            d2.loc[row, attr] = quality_label(fill.get((deal, attr)))

# The exact rate behind every label, for whoever wants the number.
d2_fill_rates = (
    pd.Series(fill, name="fill_rate").rename_axis(["deal_id", "attribute"]).reset_index()
)
print(f"{(d2_fill_rates.fill_rate >= 0.9).sum():,} of {len(d2_fill_rates):,} found attributes "
      f"are 90%+ filled; {(d2_fill_rates.fill_rate == 0).sum():,} are entirely empty")
with pd.option_context("display.float_format", lambda v: f"{v:,.5f}"):
    display(d2[d2.deal_id.isin(in_folder)])

Per attribute: how many deals have it found and **mainly filled**, against found but **mainly empty**.

In [ ]:
filled = d2[d2.deal_id.isin(in_folder)][ATTRIBUTES]
quality_summary = pd.DataFrame({
    "mainly_not_null": (filled == "Mainly Not Null (90%+)").sum(),
    "mainly_null": (filled == "Mainly Null (10%+)").sum(),
    "of_which_entirely_empty": d2_fill_rates[d2_fill_rates.fill_rate == 0].attribute.value_counts(),
    "not_found": filled.isin(["Not Found", "Unreadable"]).sum(),
}).fillna(0).astype(int).sort_values("mainly_not_null", ascending=False)
quality_summary[quality_summary.mainly_not_null + quality_summary.mainly_null > 0]

## Section E — Post-Registration Data (Snowflake)

The same question as Section D, asked of what the servicer sends us **after** we own the
book, at case level. Every deal we own is a row, and every attribute of Sections A and B a
column, as in D.1.

**Everything here is asked at case level**: do we have the attribute for each case? So
E.1 and E.2 count only case-level tables. PAM, Intrum's monthly report per batch, cannot
answer that. It appears in E.3 instead, as the yardstick CAT's collections are checked
against.

### E.1 What the raw feed holds

**RAW** is Intrum's case-level feed as it lands: CAT (`RAW.HYPERION_INTRUM_CAT`, 41
tables).

- **Found in a Table**: a CAT table has a column for the attribute, and CAT holds cases for
  this deal.
- **Not Found in a Table**: CAT has no column for it (including what only PAM reports,
  such as the deal flow type), or CAT holds nothing for this deal. The 12 GCB, LEG and ORG
  deals are serviced by Gescobro, Cerved, Massey and Fire, not Intrum, so they are never in
  CAT.
- **Not Relevant**: a Section A attribute of another country, as in D.

Which column carries each attribute is written out in
[`post_registration_map.py`](post_registration_map.py), so any call can be checked.


In [ ]:
from pathlib import Path

import post_registration_map as prmap

CACHE_DIR = Path(".cache")
REFRESH = False  # True re-runs the slow Snowflake queries instead of reading .cache/


def cached(name, query):
    """A query's result, from .cache/ unless REFRESH. Ids, counts and rates only."""
    path = CACHE_DIR / f"{name}.csv"
    if REFRESH or not path.exists():
        df = datalake.q(query)
        df.columns = df.columns.str.lower()
        CACHE_DIR.mkdir(exist_ok=True)
        df.to_csv(path, index=False)
    stamp = pd.Timestamp.fromtimestamp(path.stat().st_mtime)
    print(f"{name}: {'queried' if REFRESH else 'cached'} {stamp:%Y-%m-%d %H:%M}")
    return pd.read_csv(path)


# Which Intrum deals raw CAT has cases for: scans 6.5bn rows, so cached.
raw_cat = cached("raw_cat_deals", "work/workshop/queries/raw_cat_deals")
# Cases that were already paying before the servicer registered them: the one thing
# the D and E grids cannot tell us, since both score collections with the same metric.
pre_coll = cached("pre_registration_collections",
                  "work/workshop/queries/pre_registration_collections")
# How old the debt actually is. The grids only say whether a default date exists.
debt_age = cached("debt_age", "work/workshop/queries/debt_age")
servicer_ids = datalake.q("work/workshop/queries/deal_servicer_ids")
servicer_ids.columns = servicer_ids.columns.str.lower()

in_cat = set(servicer_ids[servicer_ids.intrum_id.isin(raw_cat.intrum_id)].deal_id)
E_ATTRIBUTES = ATTRIBUTES


def relevant(deal, attr):
    return not (attr in countries_of and country.get(deal) not in countries_of[attr])


def e1_cell(deal, attr):
    if not relevant(deal, attr):
        return "Not Relevant"
    present = prmap.raw_feed(attr) == "CAT" and deal in in_cat
    return "Found in a Table" if present else "Not Found in a Table"


e1 = deals[["deal_id", "vehicle", "country"]].rename(columns={"country": "deal_country"})
e1 = e1.assign(**{a: [e1_cell(d, a) for d in e1.deal_id] for a in E_ATTRIBUTES})
print(f"{len(e1):,} deals we own; raw CAT holds cases for {len(in_cat):,} of them")
e1

### E.2 What NPL_COMMON holds, after dbt has cleaned it

The same grid, asked of `PROD.NPL_COMMON`, the cleaned tables everything downstream reads.
Where the attribute is missing there, the cell says why:

- **Found in a Table**: an NPL_COMMON table carries it and has rows for this deal.
- **Not Found: Batch Level Only**: NPL_COMMON has it, but only per batch (`DIM_BATCH`,
  `DIM_PORTFOLIO`) or per batch and month (PAM), not per case. Examples are the seller,
  servicer, deal flow type and Intrum's capex date.
- **Not Found: DBT Clean Up Required**: the raw feed has it for this deal, but dbt does not
  carry it into NPL_COMMON. Fixing that is modelling work for the data team, not something
  to patch here.
- **Not Found: Not Sent by Servicer**: neither the raw feed nor NPL_COMMON has a column for it.
- **Not Found: No Data for Deal**: the column exists, but holds nothing for this deal.
- **Not Relevant**: as in E.1.


In [ ]:
# Fill rates and the CAT-vs-PAM reconciliation per deal: ~5 minutes, so cached.
quality = cached("post_registration_quality", "work/workshop/queries/post_registration_quality")
q = quality.set_index("deal_id")

has_rows = {
    "case": set(q.index[q.npl_cases.fillna(0) > 0]),
    "batch": set(q.index),
    "collections": set(q.index[q.cat_collection_fill.notna()]),
    "pam": set(q.index[q.pam_collection_fill.notna()]),
}


def e2_cell(deal, attr):
    if not relevant(deal, attr):
        return "Not Relevant"
    grain = prmap.npl_grain(attr)
    if grain in prmap.NOT_CASE_LEVEL:
        return "Not Found: Batch Level Only"
    if grain:
        return "Found in a Table" if deal in has_rows[grain] else "Not Found: No Data for Deal"
    if e1_cell(deal, attr) == "Found in a Table":
        return "Not Found: DBT Clean Up Required"
    if prmap.raw_feed(attr) is None:
        return "Not Found: Not Sent by Servicer"
    return "Not Found: No Data for Deal"


e2 = e1[["deal_id", "vehicle", "deal_country"]].copy()
e2 = e2.assign(**{a: [e2_cell(d, a) for d in e2.deal_id] for a in E_ATTRIBUTES})
e2

### E.3 Data quality in NPL_COMMON, and whether CAT reconciles with PAM

Same grid. Where E.2 found the attribute, the cell says how full it is, as in D.2:
**`Mainly Not Null (90%+)`** if at least 90% of the deal's cases (or batches, or collection
rows) hold a value, **`Mainly Null (10%+)`** otherwise. Other cells keep E.2's label.

Three columns at the front then compare the two collection feeds over the months both
cover:

- **`months_ratio_cat_to_pam`**: months with collections in CAT (`FCT_CASE_COLLECTION`) ÷
  months with collections in PAM (`FCT_PORTFOLIO_SNAPSHOT_MONTHLY`).
- **`collections_ratio_cat_to_pam`**: CAT's collections ÷ PAM's, over the same months.
- **`reconciliation_flag`**: **True** when the months ratio is at least 0.8 **and** the
  collections ratio is between 0.9 and 1.1.

**Why many old books fail the flag:** NPL_COMMON's case collections start where each case's
first record in Intrum's CAT system does. For books bought before about 2014 that is a
system migration date, not the registration date, so years of payments are missing. One
measured example is the Portugal / Millennium books, where case-level collections came to
42% of PAM's. This is a known dbt finding, reported to the data team and not patched here.
The window above only compares months both feeds cover, but CAT can still be short inside
that window.


In [ ]:
e2_by_deal = e2.set_index("deal_id")


def e3_cell(deal, attr):
    label = e2_by_deal.at[deal, attr]
    if label != "Found in a Table":
        return label
    rate = q.loc[deal, prmap.NPL[attr][1]]
    if pd.isna(rate):
        return "Not Found: No Data for Deal"
    return "Mainly Not Null (90%+)" if rate >= 0.9 else "Mainly Null (10%+)"


RECONCILE = ["months_ratio_cat_to_pam", "collections_ratio_cat_to_pam", "reconciliation_flag"]
e3 = e2[["deal_id", "vehicle", "deal_country"]].merge(
    quality[["deal_id"] + RECONCILE], on="deal_id", how="left"
)
e3["reconciliation_flag"] = e3.reconciliation_flag.fillna(False).astype(bool)
e3 = e3.assign(**{a: [e3_cell(d, a) for d in e3.deal_id] for a in E_ATTRIBUTES})

both = e3.months_ratio_cat_to_pam.notna()
print(f"{both.sum():,} deals have collections in both CAT and PAM; "
      f"{e3.reconciliation_flag.sum():,} reconcile "
      f"({e3.reconciliation_flag.sum() / both.sum():.5f})")
with pd.option_context("display.float_format", lambda v: f"{v:,.5f}"):
    display(e3)

Reconciliation by vehicle, and how each attribute fares across E.1 → E.2 → E.3:

In [ ]:
with pd.option_context("display.float_format", lambda v: f"{v:,.5f}"):
    display(
        e3[both].groupby("vehicle").agg(
            deals=("deal_id", "size"),
            reconcile=("reconciliation_flag", "sum"),
            median_months_ratio=("months_ratio_cat_to_pam", "median"),
            median_collections_ratio=("collections_ratio_cat_to_pam", "median"),
        )
    )

intrum = e1.deal_id.isin(in_cat)
funnel = pd.DataFrame({
    "raw_cat_found": (e1[intrum][E_ATTRIBUTES] == "Found in a Table").sum(),
    "npl_common_found": (e2[intrum][E_ATTRIBUTES] == "Found in a Table").sum(),
    "npl_common_mainly_not_null": (e3[intrum][E_ATTRIBUTES] == "Mainly Not Null (90%+)").sum(),
    "npl_common_dbt_clean_up_required": (e2[intrum][E_ATTRIBUTES] == "Not Found: DBT Clean Up Required").sum(),
}).sort_values(["raw_cat_found", "npl_common_found"], ascending=False)
funnel

## Section F — The Workbook

Everything above, written to one Excel file: [`data_landscape.xlsx`](data_landscape.xlsx),
beside this notebook. Colour-coded by tab:

| Tabs | |
|---|---|
| **Summary** | three figures, then the findings |
| **1a, 1b** | every deal, and the same regrouped by market |
| **2a, 2b** | every attribute per country, and the same per market |
| **A–E** | the evidence: attributes (A, B), the book (C), the seller's tape (D), the servicer and our warehouse (E) |
| **Guide, Data Dictionary** | contents, label legend, every column defined |

A **market** is one country's one product (`SE · Bank Loan`), and a **sector** is that
product read as bank-like or not: Financial Services where the debt began as credit,
Non-Financial Services where it began as an unpaid bill. PAM's catch-all product `Other`
is 745 deals and cannot be read either way, so it is Unclassified rather than guessed at.

Two rules run through every page. An attribute found on **either** side counts as found —
it does not matter whether the seller's tape or the servicer's feed carries it. The
exception is **payment history from before registration**, which the servicer's feed almost
never has, so it gets its own column. Everything is counted in deals and attributes, not
percentages.

Every figure is computed from the tables above when this cell runs, never typed in.
Rerun the notebook and the workbook follows.


In [ ]:
import landscape_workbook as lw

AS_OF = f"{pd.Timestamp.now():%Y-%m-%d}"
ATTR_COLS = {
    "country": "Country the attribute applies to; All for Section B.",
    "attribute": "The attribute's name. Section A names it the way that country's market does.",
    "static_or_variable": "static: fixed once known, or measured at a fixed date. variable: can change while we own the loan.",
    "local_term": "What the column is usually called on a tape: the local word (A), or Swedish · German · Portuguese · Italian · Spanish (B). (derived) = built from other columns.",
    "meaning": "What the attribute is, in plain words.",
    "why_it_matters": "Why an NPL buyer cares about it.",
    "legal_source": "A: web <date> = checked against legal sources that day; general = standard practice, not checked line by line. Not legal advice. B: Not relevant.",
    "used_in_npl_underwriting": "Whether npl-underwriting builds and uses it, measured 2026-11-02 over its 18 live deals, with the deals named.",
}
GRID_KEYS = {
    "deal_id": "VEHICLE-ID. Heimdall deals use Intrum's id; every other vehicle our Atlas id.",
    "vehicle": "HEI Heimdall, LOK Lokki, LEG legacy books, GCB, ORG.",
    "deal_country": "The deal's country (GB for the United Kingdom).",
}
ratio = "A decimal: 0.098 = 9.8%."

c_frame = deals[["deal_id", "vehicle", "country", "product_type", "sector", "market", "acquired_by",
                 "deal_flow_type", "registration_date",
                 "capex_date", "currency", "purchase_price_local", "purchase_price_eur", "cases_acquired",
                 "balance_at_acquisition_local", "balance_at_acquisition_eur", "balance_basis", "rate_to_eur", "batches"]]
e3_recon = quality[["deal_id", "npl_cases", "window_start", "window_end", "cat_months", "pam_months",
                    "cat_collected", "pam_collected", "months_ratio_cat_to_pam",
                    "collections_ratio_cat_to_pam", "reconciliation_flag"]]
for col in ("window_start", "window_end"):
    e3_recon = e3_recon.assign(**{col: pd.to_datetime(e3_recon[col])})
column_map = pd.DataFrame(
    [(a, prmap.RAW.get(a), (prmap.NPL.get(a) or (None,))[0], prmap.npl_grain(a)) for a in E_ATTRIBUTES],
    columns=["attribute", "raw_cat_column", "npl_common_column", "npl_common_grain"],
)

sheets = [
    lw.Sheet("A Country Attributes", "A — Country Attributes",
             "The main legal attributes an NPL buyer tracks in each country, named the way that country names them.",
             "One row per (country, attribute).", "General knowledge and legal sources (see legal_source); npl-underwriting code for usage.",
             country_attributes, ATTR_COLS, wide=("meaning", "why_it_matters", "used_in_npl_underwriting", "local_term")),
    lw.Sheet("B General Attributes", "B — General Attributes",
             "The attributes that matter on every tape, whatever the country.",
             "One row per attribute; country = All.", "General knowledge; npl-underwriting code for usage.",
             general_attributes, ATTR_COLS, wide=("meaning", "why_it_matters", "used_in_npl_underwriting", "local_term")),
    lw.Sheet("C Deals We Own", "C — Deals We Own",
             "Every deal Brocc owns today, and its FIRST purchase: price, capex, cases and balance at acquisition.",
             "One row per deal.", "Live query on PROD NPL_COMMON (queries/deals_we_own.sql).",
             c_frame, {
                 **GRID_KEYS, "country": "The deal's country (GB for the United Kingdom).",
                 "sector": "Financial Services (bank-like: the debt began as credit), Non-Financial Services (it began as an unpaid bill), or Unclassified for PAM's catch-all product 'Other'.",
                 "product_type": "PAM's own product (BRANCH_DESC), 14 of them. Empty on the 12 LEG, GCB and ORG books PAM never reported. A deal bought across two products takes the one holding the most principal (13 deals).",
                 "market": "Country \u00d7 sector, the two joined: the market the deal sits in. 39 of them. Never empty - a deal PAM reports no product for lands in Unclassified.",
                 "acquired_by": "Whose purchase the acquisition columns describe: Intrum (Heimdall), Brocc (Lokki), previous owner (legacy books), Brocc (GCB, ORG).",
                 "deal_flow_type": "One-off (sold once) or Forward (new defaults sold every period).",
                 "registration_date": "First case registered on or after the capex month. Empty where every case predates capex or none is loaded.",
                 "capex_date": "Date of the purchase described (see acquired_by).",
                 "currency": "Currency of the _local columns.",
                 "purchase_price_local": "Price paid in that purchase, in the deal's currency.",
                 "purchase_price_eur": "The same in EUR, at the latest month-end PMI rate. Only EUR columns may be added across deals.",
                 "cases_acquired": "Cases bought in that purchase. Empty where Atlas is the source (GCB, ORG): it holds no count.",
                 "balance_at_acquisition_local": "Balance bought, in the deal's currency; balance_basis says what it includes.",
                 "balance_at_acquisition_eur": "The same in EUR.",
                 "balance_basis": "principal (PAM and servicer seeds hold nothing else at acquisition) or gbv: principal + interest + fees (Atlas).",
                 "rate_to_eur": "EUR per unit of the deal's currency, latest month-end PMI rate.",
                 "batches": "Servicer batches the deal is reported in.",
             }, money=("purchase_price_local", "purchase_price_eur", "balance_at_acquisition_local", "balance_at_acquisition_eur"),
             decimals=("rate_to_eur",)),
    lw.Sheet("D1 Seller Tape Found", "D.1 — Pre-registration: is it in the seller's files?",
             "For every deal, whether the seller's own files carry each attribute, and whether that tape links to the servicer's cases.",
             "One row per deal; one column per attribute of Sections A and B.", "Headers of every sheet of every seller file in work/workshop/asset-index/ (values not read).",
             d1, {**GRID_KEYS, "bridge_coverage": "Found when the deal's _meta/bridge.yaml names a tape column matching Intrum's case references; Not Found when none, or there is no bridge.yaml.",
                  "(attribute columns)": "Found / Not Found / Unreadable / Not Relevant / No Seller Tape. See Guide."}, grid=True),
    lw.Sheet("D2 Seller Tape Filled", "D.2 — Pre-registration: is it filled?",
             "Where D1 found an attribute, whether the column is filled on at least 90% of the tape's rows.",
             "One row per deal; one column per attribute.", "Every row of every matched sheet, read in memory; only rates kept (cached in .cache/).",
             d2, {**GRID_KEYS, "bridge_coverage": "Share of Intrum's cases the tape reaches, from bridge.yaml. " + ratio,
                  "(attribute columns)": "Mainly Not Null (90%+) / Mainly Null (10%+), or D.1's label where not found. See Guide."},
             grid=True, decimals=("bridge_coverage",)),
    lw.Sheet("D3 Fill Rates", "D.2 — The rate behind every label",
             "The exact rate behind every filled / not filled label.",
             "One row per (deal, attribute found).", "As D.2.",
             d2_fill_rates, {"deal_id": GRID_KEYS["deal_id"], "attribute": "The attribute.",
                             "fill_rate": "Share of the tape's data rows holding a real value, in the fullest matching column. Derived attributes: their emptiest input. " + ratio},
             decimals=("fill_rate",)),
    lw.Sheet("D4 Evidence", "D.1 — Why each Found was found",
             "The header and the file behind every Found, so any call can be checked.",
             "One row per (deal, attribute found).", "As D.1. Header names only, no values.",
             d1_evidence, {"deal_id": GRID_KEYS["deal_id"], "attribute": "The attribute.",
                           "headers": "Up to three headers that matched, or (derived from ...) for a derived attribute.",
                           "example_file": "The first file the headers were found in."}, wide=("headers", "example_file")),
    lw.Sheet("D5 Unreadable Files", "D — Files that could not be read",
             "Seller files nobody could open, so their columns are unknown.",
             "One row per file.", "As D.1.",
             unreadable, {"deal_id": GRID_KEYS["deal_id"], "file": "The file, inside the deal's folder.",
                          "error": "Why. DRM-protected = encrypted with rights management; only Excel with access opens it."}, wide=("file", "error")),
    lw.Sheet("E1 Servicer Feed", "E.1 — Post-registration: is it in Intrum's raw feed?",
             "For every deal, whether the servicer's case-level feed carries each attribute.",
             "One row per deal; one column per attribute. Case level only: PAM, a per-batch report, does not count.",
             "RAW.HYPERION_INTRUM_CAT, every CASES extract (queries/raw_cat_deals.sql, cached); columns mapped in post_registration_map.py.",
             e1, {**GRID_KEYS, "(attribute columns)": "Found in a Table / Not Found in a Table / Not Relevant. See Guide."}, grid=True),
    lw.Sheet("E2 Warehouse", "E.2 — Post-registration: is it in NPL_COMMON?",
             "The same grid asked of our warehouse, and where it is missing, why.",
             "One row per deal; one column per attribute. Case level only.",
             "PROD NPL_COMMON (queries/post_registration_quality.sql, cached).",
             e2, {**GRID_KEYS, "(attribute columns)": "Found in a Table, or Not Found with the reason. See Guide."}, grid=True),
    lw.Sheet("E3 Warehouse Filled", "E.3 — Post-registration: is it filled, and does CAT reconcile with PAM?",
             "Where E2 found an attribute, whether 90%+ of the deal's cases hold it, and whether its collections match the servicer's own report.",
             "One row per deal; one column per attribute.", "As E.2.",
             e3, {**GRID_KEYS,
                  "months_ratio_cat_to_pam": "Months with collections in CAT (FCT_CASE_COLLECTION) / in PAM (FCT_PORTFOLIO_SNAPSHOT_MONTHLY), over the months both cover.",
                  "collections_ratio_cat_to_pam": "CAT's collections / PAM's, over the same months.",
                  "reconciliation_flag": "TRUE when months ratio >= 0.8 AND collections ratio is between 0.9 and 1.1.",
                  "(attribute columns)": "Mainly Not Null (90%+) / Mainly Null (10%+), or E.2's label. See Guide."},
             grid=True, decimals=("months_ratio_cat_to_pam", "collections_ratio_cat_to_pam")),
    lw.Sheet("E4 Reconciliation", "E.3 — The numbers behind the reconciliation",
             "Our collections against the servicer's own report, over the months both cover.",
             "One row per deal.", "As E.2. Collections dated after today (841 rows, to the year 4570) are excluded.",
             e3_recon, {"deal_id": GRID_KEYS["deal_id"],
                        "npl_cases": "Cases the deal has in NPL_COMMON's case snapshot.",
                        "window_start": "First month both feeds cover.", "window_end": "Last month both feeds cover.",
                        "cat_months": "Months in the window with case collections (CAT).", "pam_months": "Months in the window with PAM collections.",
                        "cat_collected": "Case collections in the window, in the deal's currency.",
                        "pam_collected": "PAM gross cash recoveries in the window, in the deal's currency.",
                        "months_ratio_cat_to_pam": "cat_months / pam_months.", "collections_ratio_cat_to_pam": "cat_collected / pam_collected.",
                        "reconciliation_flag": "TRUE when months ratio >= 0.8 AND collections ratio is between 0.9 and 1.1."},
             money=("cat_collected", "pam_collected"), decimals=("months_ratio_cat_to_pam", "collections_ratio_cat_to_pam")),
    lw.Sheet("E5 Column Map", "E — Which column carries each attribute",
             "Which source column carries each attribute, behind every E1 and E2 call.",
             "One row per attribute.", "post_registration_map.py, checked against describe table on 2026-11-03.",
             column_map, {"attribute": "The attribute.",
                          "raw_cat_column": "Raw CAT table.column, or empty where CAT has none at case level.",
                          "npl_common_column": "NPL_COMMON table.column, or empty where dbt does not carry it.",
                          "npl_common_grain": "case, collections, batch or pam. batch and pam are not case level, so count as Batch Level Only."},
             wide=("raw_cat_column", "npl_common_column")),
]

# --- the Summary: every figure from the tables above ---
in_folder_set = set(in_folder)
filled90 = (d2_fill_rates.fill_rate >= 0.9).sum()
bridged = sum(v is not None for v in coverage_pct.values())
recon_n, both_n = int(e3.reconciliation_flag.sum()), int(both.sum())
dbt_attrs = sorted(a for a in E_ATTRIBUTES if (e2[a] == "Not Found: DBT Clean Up Required").any())
summary_rows = [
    ("Deals we own", f"{len(deals):,}", "Solid: live PROD query.", "C Deals We Own"),
    ("Deals the servicer sends case-level data for", f"{len(in_cat):,} of {len(deals):,}",
     "Solid: full scan of every extract.", "E1 Servicer Feed"),
    ("Deals whose collections match the servicer's own report", f"{recon_n:,} of {both_n:,}",
     "Solid as a measure; why each deal fails is not yet explained.", "E4 Reconciliation"),
    ("Deals we hold a seller tape for", f"{len(in_folder_set)} of {len(deals):,}",
     "Solid: counted from the folders.", "D1 Seller Tape Found"),
    ("Tapes that link to the servicer's cases", f"{bridged} of {len(in_folder_set)}",
     "Solid.", "D2 Seller Tape Filled"),
    ("Seller attributes found and 90%+ filled", f"{filled90:,} of {len(d2_fill_rates):,}",
     "Solid on Found; shaky on Not Found, where a header can be missed.", "D3 Fill Rates"),
    ("Attributes the servicer sends but the warehouse drops", f"{len(dbt_attrs)}",
     "Solid: checked against the schema.", "E2 Warehouse"),
    ("Seller files nobody can open", f"{len(unreadable)} files, {len(locked)} deals",
     "Solid.", "D5 Unreadable Files"),
    ("Collection rows dated in the future", "841 rows, to the year 4570",
     "Solid: excluded from every figure here.", "E4 Reconciliation"),
]

# One line per sheet for the Contents. Short on purpose: the sheet itself says more.
NAV = {
    "1a Deal Readiness": "Every deal: what it clears, how many attributes we can use, and whether we have its pre-registration payments.",
    "1b By Market (R)": "Every market: how many deals we can follow from the day we bought them.",
    "1b By Market (F)": "Every market: how many we can follow from origination. A subset of (R).",
    "2a By Attribute": "Every attribute, per country: on how many of its deals we can use it.",
    "2b By Market and Attribute (R)": "Every attribute, in every market, over the post-registration deals.",
    "2c By Market and Attribute (F)": "The same, over the full-timeline deals. Only 8 deals - read the counts.",
    "A Country Attributes": "The legal attributes each country's market tracks.",
    "B General Attributes": "The attributes that matter on every tape.",
    "C Deals We Own": "Every deal, its market, and what we paid.",
    "D1 Seller Tape Found": "Before we bought it: is the attribute in the seller's files?",
    "D2 Seller Tape Filled": "And is the column actually filled?",
    "D3 Fill Rates": "The exact rate behind every filled / not filled label.",
    "D4 Evidence": "The header and file behind every Found.",
    "D5 Unreadable Files": "Seller files nobody could open.",
    "E1 Servicer Feed": "After we bought it: does the servicer send the attribute?",
    "E2 Warehouse": "Does it reach our warehouse per case, and if not, why?",
    "E3 Warehouse Filled": "And is it filled?",
    "E4 Reconciliation": "Our collections against the servicer's own report, per deal.",
    "E5 Column Map": "Which source column carries each attribute.",
}
note = ("Grid sheets (D1, D2, E1, E2, E3) have one column per attribute of Sections A and B, "
        "defined on sheets A and B; their labels are defined on the Guide sheet.")
# --- the two pages the workbook leads with, in front of the evidence ---
# Built from the label grids above. An attribute found on either side counts as found;
# payment history from before registration is the exception and has its own column.
import importlib

import landscape_funnel as lf
importlib.reload(lf)
sheets = lf.all_sheets(deals, d2, e3, quality, in_folder, coverage_pct, in_cat,
                       ATTRIBUTES, countries_of, country, pre_coll, debt_age) + sheets

# The three figures on the front page, read off the deal page so they cannot disagree.
gates = lf.deal_funnel_frame(sheets[0].frame)
own, served, modelable = (int(gates.deals.iloc[i]) for i in (0, 1, -1))
headline = [
    ("deals we own", f"{own:,}", "Every book in the portfolio."),
    ("the servicer sends case data for", f"{served:,}", f"{served / own:.1%} of what we own."),
    ("we can model end to end", f"{modelable:,}",
     f"{modelable / own:.1%}: seller tape and servicer feed, joined."),
]

out = Path("data_landscape.xlsx")
lw.build(out, sheets, summary_rows, AS_OF, note, headline=headline, nav=NAV)
print(f"wrote {out} ({out.stat().st_size / 1e6:.1f} MB): Summary, the deal and attribute "
      f"pages, {len(sheets) - 2} evidence sheets, Guide and Data Dictionary")

# --- the dashboard's funnel ---------------------------------------------------------
# Written out for the NPL Modelling Readiness dashboard, not for the workbook. Same
# masks as pages 1b and 2b/2c, so the figures cannot drift from them.
_relevant = lf._relevance(country, ATTRIBUTES, countries_of)
_, _use_pre = lf._masks(d2, ATTRIBUTES)
_, _raw_post = lf._masks(e3, ATTRIBUTES)
_use_post = lf._correct_pre_payments(_raw_post, lf._feed_pays(pre_coll))
_relevant = _relevant.reindex(_use_pre.index)
_stages, _gate_attr = lf.gate_frames(sheets[0].frame, _relevant, _use_pre, _use_post, ATTRIBUTES)
import pathlib as _pl
_out = _pl.Path("/tmp/dash")
_out.mkdir(exist_ok=True)
_out.joinpath("funnel.json").write_text(_stages.to_json(orient="records"))
_out.joinpath("gate_attr.json").write_text(_gate_attr.to_json(orient="records"))
# Per-market funnel, so the dashboard can filter the Overview to one market.
_fm = lf.gate_market_frame(sheets[0].frame)
_out.joinpath("funnel_market.json").write_text(_fm.to_json(orient="records"))
# Where each attribute can come from, for the dashboard's colour coding.
_src = pd.DataFrame({"attribute": ATTRIBUTES})
_src["source"] = [lf.attribute_source(a) for a in ATTRIBUTES]
_out.joinpath("attribute_source.json").write_text(_src.to_json(orient="records"))
print("funnel_market", len(_fm), "rows |", _src.source.value_counts().to_dict())
print("funnel stages", len(_stages), "| gate x attribute rows", len(_gate_attr))
print(_stages.to_string(index=False))
